# مدل عیب‌های لحیم‌کاری و مونتاژ — YOLO26 روی Google Colab

**چه چیزی پیدا می‌کند:** لحیم کم، لحیم زیاد، پل لحیم، حفره‌ی لحیم، قطعه‌ی جاافتاده، قطعه‌ی جابه‌جا، سوختگی، آسیب فیزیکی، اکسید

## اجرا (فقط همین)
1. کل پوشه‌ی `1_Solder_Model` را در Google Drive (داخل My Drive) آپلود کن: این نوت‌بوک + `solder_data.zip`.
2. نوت‌بوک را با Google Colaboratory باز کن ▸ `Runtime ▸ Change runtime type ▸ T4 GPU ▸ Save`
3. `Runtime ▸ Run all` و اجازه‌ی دسترسی به Google Drive را بده.
4. **قطع شد یا لیمیت خوردی؟** فقط دوباره `Run all` (با همین حساب یا حساب دیگری که پوشه را دارد): از آخرین نسخه‌ی ذخیره‌شده ادامه می‌دهد.

| بلوک | کار |
|---|---|
| ⚙️ تنظیمات | مدل، تعداد epoch، اندازه‌ی تصویر (پیش‌فرض‌ها برای بیشترین دقت انتخاب شده‌اند) |
| ۱، ۲ | آماده‌سازی Colab و داده (هر جلسه چند دقیقه) |
| **۳) وضعیت آموزش** | قبل از شروع: چند epoch انجام شده، بهترین دقت، زمان باقی‌مانده و این‌که آموزش از کجا ادامه پیدا می‌کند |
| **۴) آموزش** | پنل زنده که هر ۲ ثانیه به‌روز می‌شود: نوار پیشرفت کل و epoch جاری، loss، دقت هر epoch، نمودار، زمان باقی‌مانده، GPU |
| ۵، ۶ | نتیجه روی بردهای دیده‌نشده (هر کلاس جدا) و تست روی عکس خودت |

**توقف دستی:** دکمه‌ی ■ (Interrupt). آخرین epoch کامل‌شده روی Drive ذخیره می‌شود و دفعه‌ی بعد از همان‌جا ادامه می‌دهد.

<details open><summary>تنظیمات پایداری و دقت (نسخه‌ی v3)</summary>

در آموزش قبلی (v2) بعد از epoch 4 دقت از ۰٫۱۸ (mAP50-95) به ۰٫۰۵ افتاد و دیگر بالا نیامد. علت **بیش‌برازش یا کمبود داده نبود**:
loss در epoch 5 ناگهان منفجر شد (cls از ۲٫۷ به ۲۶) و مدل هیچ‌وقت برنگشت. دو تنظیم v2 با پیش‌فرض Ultralytics فرق داشتند
(نرخ یادگیری ۰٫۰۰۱ به‌جای ۰٫۰۰۰۷۷ و `warmup_bias_lr=0.1` که برای AdamW بسیار زیاد است). در v3:
- نرخ یادگیری **۰٫۰۰۰۵**، `warmup_bias_lr=0`، گرم‌کردن **۵ epoch**.
- **محافظ آموزش:** بعد از هر epoch چک می‌کند loss منفجر نشده و دقت نیفتاده. اگر شد، خودکار به **بهترین وزن** برمی‌گردد،
  نرخ یادگیری را نصف می‌کند و در پنل می‌نویسد چه شد. بعد از ۴ بار ناپایداری آموزش را متوقف می‌کند (ساعت‌ها هدر نمی‌رود).
- augmentation واقعی‌تر: نویز دوربین، محوشدگی حرکتی و فشرده‌سازی JPEG (روی تصویر، کادرها دست‌نخورده) + مقیاس بیشتر.
  کپی آفلاین از عکس‌ها فایده‌ای ندارد (همان augmentation تصادفی هر epoch را انجام می‌دهد)؛ تنوع واقعی فقط با عکس واقعی بیشتر می‌آید.
- وزن بهترین epoch آموزش‌های قبلی خودکار در `saved_weights` ذخیره می‌شود. آموزش v3 در پوشه‌ی جدید خودش است.

</details>

<details><summary>چه چیزهایی در نسخه‌ی v2 برای دقت و سرعت بهتر شده بود</summary>

- **مدل بزرگ‌تر: YOLO26m** (۲۰ میلیون پارامتر، به‌جای ۱۰ میلیون در s). Colab رایگان فقط ۲ هسته‌ی CPU دارد و در آموزش قبلی GPU بخشی از وقت منتظر CPU بود؛ مدل بزرگ‌تر از همان GPU بیشتر استفاده می‌کند.
- **150 epoch** با برنامه‌ی کاهش نرم نرخ یادگیری (cosine) و AdamW؛ چند epoch آخر بدون موزاییک (روی عکس‌های کامل) برای دقت نهایی.
  هر epoch هر عکس train را دو بار می‌بیند (خود عکس + نسخه‌ی ۹۰ درجه).
  اگر 50 epoch پشت سر هم بهتر نشود، زودتر تمام می‌شود (patience).
- **batch ثابت و درست** به اندازه‌ی حافظه‌ی GPU (AutoBatch قبلی batch خیلی کوچکی انتخاب می‌کرد و GPU نیمه‌خالی می‌ماند).
- عکس‌های بزرگ train (برش‌های ۱۲۸۰ پیکسلی neazu) **یک بار با کیفیت بالا (INTER_AREA)** به اندازه‌ی آموزش کوچک می‌شوند:
  بارگذاری سریع‌تر و جزئیات ریز تمیزتر.
- آموزش روی دیسک سریع Colab انجام می‌شود و نسخه‌ی امن **هر ۲ epoch**، هنگام توقف دستی و در پایان روی Drive کپی می‌شود.
- **حداکثر سرعت:** batch به اندازه‌ی حافظه‌ی GPU (روی GPUهای بزرگ‌تر خودکار بیشتر)، تصاویر در RAM کش می‌شوند اگر جا باشد
  (وگرنه روی دیسک سریع)، همه‌ی هسته‌های CPU برای بارگذاری داده، cuDNN autotune، AMP.
- **GPU نبود؟** خودکار روی CPU اجرا می‌شود (خیلی کندتر؛ فقط برای آزمایش مناسب است).
- **آستانه‌ی اطمینان جدا برای هر کلاس** روی val تنظیم می‌شود (بیشترین F1) و پیش‌بینی‌ها با همان آستانه‌ها انجام می‌شود.
- انتخاب بهترین مدل فقط با **val**؛ test فقط برای گزارش صادقانه است.

زمان تقریبی روی T4 رایگان: حدود ۴ تا ۵ دقیقه برای هر epoch با YOLO26m (چند جلسه‌ی Colab؛ ادامه خودکار است). YOLO26s تقریباً دو برابر سریع‌تر ولی کمی کم‌دقت‌تر است. عدد دقیق بعد از اولین epoch در پنل نشان داده می‌شود.
</details>

<details><summary>گزارش دیتاست</summary>

```
Soldering / assembly defects (after assembly)
=============================================
classes: ['insufficient_solder', 'excess_solder', 'solder_bridge', 'solder_void', 'missing_component', 'shifted_component', 'burn', 'physical_damage', 'oxidation']

source              images  groups  train/val/test images
sssssc_assembly        826     487  619/103/104
numeric123             762     175  571/96/95
neazu_solder           158     157  118/20/20
zurab_bga              166     143  166/0/0
vattipalli              78      46  64/7/7
cv_solder_splash        29       6  23/3/3

split   files  empty insufficient_s  excess_solder  solder_bridge    solder_void missing_compon shifted_compon           burn physical_damag      oxidation
train    3836    344           2370           1550           4160            794            948            450            450           1152            830
val       687    433            187            111            181            104             74             31             31             15              0
test      686    427            186            115            181             28             74             31             32             16              0

excluded (excluded.csv has every file and its reason):
      5  mirrored/rotated duplicate
      4  train: defect-free image beyond the background cap

```
</details>

In [ ]:
#@title ⚙️ تنظیمات (پیش‌فرض‌ها برای بیشترین دقت انتخاب شده‌اند)
MODEL        = "solder"
MODEL_SIZE   = "yolo26m.pt"   #@param ["yolo26s.pt", "yolo26m.pt", "yolo26l.pt"]
EPOCHS       = 150   #@param {type:"integer"}
IMGSZ        = 800   #@param {type:"integer"}
PATIENCE     = 50   #@param {type:"integer"}
BATCH        = 0   #@param {type:"integer"}
SAVE_EVERY   = 2   #@param {type:"integer"}
QUICK_TEST   = False   #@param {type:"boolean"}
DRIVE_DIR    = "/content/drive/MyDrive/1_Solder_Model"   #@param {type:"string"}
# MODEL_SIZE  : s سریع‌تر، m دقیق‌تر (پیشنهاد)، l کمی دقیق‌تر و خیلی کندتر
# EPOCHS      : وسط آموزش هم می‌شود عوضش کرد؛ دفعه‌ی بعد که ادامه پیدا کند هدف جدید اعمال می‌شود
# PATIENCE    : اگر دقت val این تعداد epoch بهتر نشود، آموزش زودتر تمام می‌شود
# BATCH       : 0 = خودکار بر اساس مدل و حافظه‌ی GPU (روی T4: s=16، m=8، l=6 در اندازه‌ی 800)
# SAVE_EVERY  : هر چند epoch مدل روی Drive ذخیره شود (قطع شدن = از دست رفتن حداکثر همین تعداد epoch)
# QUICK_TEST  : آزمایش چنددقیقه‌ای کل مسیر (2 epoch کوچک) در پوشه‌ی جدا؛ برای آموزش اصلی خاموش باشد
RECIPE = "v3"
RUN_IMGSZ, RUN_EPOCHS = (320, 2) if QUICK_TEST else (int(IMGSZ), int(EPOCHS))
RUN_NAME = f"{MODEL_SIZE[:-3]}_{RUN_IMGSZ}_{RECIPE}"
print("مدل:", MODEL_SIZE, "| imgsz", RUN_IMGSZ, "| epochs", RUN_EPOCHS, "| patience", PATIENCE,
      "| پوشه‌ی آموزش:", RUN_NAME, "| آزمایش سریع" if QUICK_TEST else "| آموزش کامل")

## 1) آماده‌سازی: Drive، GPU، کتابخانه‌ها، کدها

In [ ]:
import os, sys, subprocess, glob, json, shutil, zipfile, base64, time, re, signal, threading, collections, importlib
WORK = "/content/work"                        # local fast disk (wiped on disconnect; results live on Drive)
os.makedirs(WORK, exist_ok=True)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

def run_live(cmd):
    """Run a short command and stream its output into the cell (progress bars collapse to their last state)."""
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    try:
        for raw in p.stdout:                  # a progress bar redraws with \r: keep only its final state
            parts = [s for s in raw.decode("utf-8", "replace").rstrip("\r\n").split("\r") if s.strip()]
            if parts:
                print(parts[-1], flush=True)
    except KeyboardInterrupt:
        p.kill()
        raise
    if p.wait() != 0:
        raise RuntimeError(f"دستور با خطا تمام شد (کد {p.returncode}) - چند خط بالاتر علت را نوشته است")

# Python packages: the exact ultralytics version this code was tested with
try:
    import ultralytics
    have = ultralytics.__version__
except ImportError:
    have = None
if have != "8.4.14":
    run_live([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.14"])
import ultralytics, torch
importlib.reload(ultralytics)
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| python", sys.version.split()[0])

if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f"GPU: {g.name} ({g.total_memory / 1e9:.0f} GB) | CPU cores: {os.cpu_count()}")
else:
    print("⚠️ GPU پیدا نشد: آموزش خودکار روی CPU انجام می‌شود (خیلی کندتر). برای سرعت: "
          "Runtime ▸ Change runtime type ▸ T4 GPU ▸ Save و دوباره Run all")
print(f"فضای آزاد دیسک Colab: {shutil.disk_usage(WORK).free / 1e9:.0f} GB")

# the scripts (verify / train / predict / live panel), written next to each other on the local disk
SCRIPTS = {"verify_dataset.py": "IiIiQ2hlY2sgZmluYWxfZGF0YXNldCBiZWZvcmUgdHJhaW5pbmcgKHJ1bnMgbG9jYWxseSBhbmQgb24gQ29sYWIsIDwgMSBtaW51dGUpLgoKSGFyZCBlcnJvcnMgKGV4aXQgY29kZSAxKTogbWlzc2luZy9pbnZhbGlkIGxhYmVsIGZpbGVzLCB2YWx1ZXMgb3V0IG9mIHJhbmdlLCB1bmtub3duIGNsYXNzIGlkLAp1bnJlYWRhYmxlIGltYWdlLCBhIGJvYXJkIGluIHR3byBzcGxpdHMsIHRoZSBzYW1lIHBpeGVscyBpbiB0d28gc3BsaXRzLCBhbiBlbXB0eSBzcGxpdC4KV2FybmluZ3M6IGNsYXNzZXMgd2l0aCBmZXcgYm94ZXMgaW4gdmFsL3Rlc3QsIHRpbnkgYm94ZXMuCgogIHB5dGhvbiB2ZXJpZnlfZGF0YXNldC5weSAtLXJvb3QgZmluYWxfZGF0YXNldAoiIiIKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IHN5cwpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyLCBkZWZhdWx0ZGljdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBjdjIKaW1wb3J0IHlhbWwKCgpkZWYgbWFpbigpOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcm9vdCIsIHJlcXVpcmVkPVRydWUpCiAgICBhID0gYXAucGFyc2VfYXJncygpCiAgICByb290ID0gUGF0aChhLnJvb3QpCiAgICBjZmcgPSB5YW1sLnNhZmVfbG9hZChvcGVuKHJvb3QgLyAiZGF0YS55YW1sIiwgZW5jb2Rpbmc9InV0ZjgiKSkKICAgIG5hbWVzID0gbGlzdChjZmdbIm5hbWVzIl0udmFsdWVzKCkpIGlmIGlzaW5zdGFuY2UoY2ZnWyJuYW1lcyJdLCBkaWN0KSBlbHNlIGNmZ1sibmFtZXMiXQogICAgZXJyLCB3YXJuLCBzdGF0cyA9IFtdLCBbXSwge30KICAgIHNlZW5fbWQ1ID0ge30KICAgIGZvciBzcCBpbiAoInRyYWluIiwgInZhbCIsICJ0ZXN0Iik6CiAgICAgICAgaW1ncyA9IHNvcnRlZCgocm9vdCAvICJpbWFnZXMiIC8gc3ApLmdsb2IoIiouanBnIikpCiAgICAgICAgY250LCBuX2JveCwgdGlueSwgZW1wdHkgPSBDb3VudGVyKCksIDAsIDAsIDAKICAgICAgICBpZiBub3QgaW1nczoKICAgICAgICAgICAgZXJyLmFwcGVuZChmIntzcH06IG5vIGltYWdlcyIpCiAgICAgICAgZm9yIGltIGluIGltZ3M6CiAgICAgICAgICAgIGxibCA9IHJvb3QgLyAibGFiZWxzIiAvIHNwIC8gKGltLnN0ZW0gKyAiLnR4dCIpCiAgICAgICAgICAgIGlmIG5vdCBsYmwuZXhpc3RzKCk6CiAgICAgICAgICAgICAgICBlcnIuYXBwZW5kKGYie3NwfTogbWlzc2luZyBsYWJlbCBmb3Ige2ltLm5hbWV9IikKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGltZyA9IGN2Mi5pbXJlYWQoc3RyKGltKSkKICAgICAgICAgICAgaWYgaW1nIGlzIE5vbmU6CiAgICAgICAgICAgICAgICBlcnIuYXBwZW5kKGYie3NwfTogdW5yZWFkYWJsZSB7aW0ubmFtZX0iKQogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgaCwgdyA9IGltZy5zaGFwZVs6Ml0KICAgICAgICAgICAgbWQ1ID0gaGFzaGxpYi5tZDUoaW1nLnRvYnl0ZXMoKSkuaGV4ZGlnZXN0KCkKICAgICAgICAgICAgaWYgbWQ1IGluIHNlZW5fbWQ1IGFuZCBzZWVuX21kNVttZDVdWzBdICE9IHNwOgogICAgICAgICAgICAgICAgZXJyLmFwcGVuZChmImlkZW50aWNhbCBpbWFnZSBpbiB7c2Vlbl9tZDVbbWQ1XVswXX0gYW5kIHtzcH06IHtzZWVuX21kNVttZDVdWzFdfSAvIHtpbS5uYW1lfSIpCiAgICAgICAgICAgIHNlZW5fbWQ1LnNldGRlZmF1bHQobWQ1LCAoc3AsIGltLm5hbWUpKQogICAgICAgICAgICBsaW5lcyA9IFtsLnNwbGl0KCkgZm9yIGwgaW4gbGJsLnJlYWRfdGV4dChlbmNvZGluZz0idXRmOCIpLnNwbGl0bGluZXMoKSBpZiBsLnN0cmlwKCldCiAgICAgICAgICAgIGVtcHR5ICs9IG5vdCBsaW5lcwogICAgICAgICAgICBmb3IgcCBpbiBsaW5lczoKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBjLCB4LCB5LCBidywgYmggPSBpbnQocFswXSksICptYXAoZmxvYXQsIHBbMTo1XSkKICAgICAgICAgICAgICAgICAgICBhc3NlcnQgbGVuKHApID09IDUKICAgICAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICAgICAgZXJyLmFwcGVuZChmIntzcH06IGJhZCBsaW5lIGluIHtsYmwubmFtZX06IHtwfSIpCiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIGlmIG5vdCAoMCA8PSBjIDwgbGVuKG5hbWVzKSkgb3Igbm90ICgwIDw9IHggPD0gMSBhbmQgMCA8PSB5IDw9IDEgYW5kIDAgPCBidyA8PSAxIGFuZCAwIDwgYmggPD0gMSk6CiAgICAgICAgICAgICAgICAgICAgZXJyLmFwcGVuZChmIntzcH06IG91dC1vZi1yYW5nZSB2YWx1ZSBpbiB7bGJsLm5hbWV9OiB7cH0iKQogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICBjbnRbbmFtZXNbY11dICs9IDEKICAgICAgICAgICAgICAgIG5fYm94ICs9IDEKICAgICAgICAgICAgICAgIHRpbnkgKz0gbWluKGJ3ICogdywgYmggKiBoKSA8IDYKICAgICAgICBzdGF0c1tzcF0gPSB7ImZpbGVzIjogbGVuKGltZ3MpLCAiYm94ZXMiOiBuX2JveCwgImVtcHR5X2ZpbGVzIjogZW1wdHksICJib3hlc191bmRlcl82cHgiOiB0aW55LAogICAgICAgICAgICAgICAgICAgICAicGVyX2NsYXNzIjoge246IGNudFtuXSBmb3IgbiBpbiBuYW1lc319CiAgICAgICAgZm9yIG4gaW4gbmFtZXM6CiAgICAgICAgICAgIGlmIHNwICE9ICJ0cmFpbiIgYW5kIGNudFtuXSA8IDE1OgogICAgICAgICAgICAgICAgd2Fybi5hcHBlbmQoZiJ7c3B9OiBjbGFzcyAne259JyBoYXMgb25seSB7Y250W25dfSBib3hlcyAoaXRzIG1ldHJpYyB3aWxsIGJlIG5vaXN5IG9yIG1pc3NpbmcpIikKICAgICAgICBpZiB0aW55ID4gMC4wNSAqIG1heCgxLCBuX2JveCk6CiAgICAgICAgICAgIHdhcm4uYXBwZW5kKGYie3NwfToge3Rpbnl9IGJveGVzIGFyZSB1bmRlciA2IHB4IikKICAgIG1hbiA9IHJvb3QgLyAic3BsaXRfbWFuaWZlc3QuY3N2IgogICAgaWYgbWFuLmV4aXN0cygpOgogICAgICAgIHdoZXJlID0gZGVmYXVsdGRpY3Qoc2V0KQogICAgICAgIGZvciByIGluIGNzdi5EaWN0UmVhZGVyKG9wZW4obWFuLCBlbmNvZGluZz0idXRmOCIpKToKICAgICAgICAgICAgd2hlcmVbci5nZXQoImdyb3VwIikgb3Igci5nZXQoImJvYXJkIildLmFkZChyWyJzcGxpdCJdKQogICAgICAgIGZvciBiLCBzIGluIHdoZXJlLml0ZW1zKCk6CiAgICAgICAgICAgIGlmIGxlbihzKSA+IDE6CiAgICAgICAgICAgICAgICBlcnIuYXBwZW5kKGYiYm9hcmQge2J9IGlzIGluIHNldmVyYWwgc3BsaXRzOiB7c29ydGVkKHMpfSIpCiAgICBlbHNlOgogICAgICAgIHdhcm4uYXBwZW5kKCJzcGxpdF9tYW5pZmVzdC5jc3YgbWlzc2luZzogYm9hcmQgbGVha2FnZSBub3QgY2hlY2tlZCIpCiAgICByZXAgPSB7Im9rIjogbm90IGVyciwgImVycm9ycyI6IGVycls6NTBdLCAibl9lcnJvcnMiOiBsZW4oZXJyKSwgIndhcm5pbmdzIjogd2FybiwgInNwbGl0cyI6IHN0YXRzfQogICAgKHJvb3QgLyAicWFfcmVwb3J0Lmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVwLCBpbmRlbnQ9MiwgZW5zdXJlX2FzY2lpPUZhbHNlKSwgZW5jb2Rpbmc9InV0ZjgiKQogICAgZm9yIHNwLCBzIGluIHN0YXRzLml0ZW1zKCk6CiAgICAgICAgcHJpbnQoZiJ7c3A6NX0gZmlsZXM9e3NbJ2ZpbGVzJ106NH0gYm94ZXM9e3NbJ2JveGVzJ106NX0gZW1wdHk9e3NbJ2VtcHR5X2ZpbGVzJ106M30ge3NbJ3Blcl9jbGFzcyddfSIpCiAgICBmb3Igd18gaW4gd2FybjoKICAgICAgICBwcmludCgiV0FSTklORzoiLCB3XykKICAgIGZvciBlIGluIGVycls6MjBdOgogICAgICAgIHByaW50KCJFUlJPUjoiLCBlKQogICAgcHJpbnQoIkRBVEFTRVQgT0siIGlmIG5vdCBlcnIgZWxzZSBmIkRBVEFTRVQgSEFTIHtsZW4oZXJyKX0gRVJST1JTIikKICAgIHN5cy5leGl0KDAgaWYgbm90IGVyciBlbHNlIDEpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=", "train_solder.py": "...", "predict.py": "IiIiRmluZCBQQ0IgZGVmZWN0cyBvbiB5b3VyIG93biBwaG90b3MgKGFuIGltYWdlIG9yIGEgZm9sZGVyKS4KCkxhcmdlIHBob3RvcyBhcmUgY3V0IGludG8gb3ZlcmxhcHBpbmcgdGlsZXMgKGFzIGluIHRyYWluaW5nKSwgZXZlcnkgdGlsZSBpcyBwcmVkaWN0ZWQsIGFuZCB0aGUKYm94ZXMgYXJlIG1lcmdlZCB3aXRoIE5NUywgc28gYSAzMDAwIHB4IHBob3RvIGtlZXBzIGl0cyBmdWxsIGRldGFpbC4gT3V0cHV0cyBwZXIgaW1hZ2U6CjxuYW1lPl9hbm5vdGF0ZWQuanBnIGFuZCA8bmFtZT4uanNvbiAoY2xhc3MsIGNvbmZpZGVuY2UsIGJveCBpbiBwaXhlbHMpOyBzdW1tYXJ5LmNzdiBmb3IgYWxsLgoKQ29uZmlkZW5jZTogYnkgZGVmYXVsdCB0aGUgcGVyLWNsYXNzIHRocmVzaG9sZHMgb2YgdGhyZXNob2xkcy5qc29uIG5leHQgdG8gdGhlIHdlaWdodHMgKHR1bmVkIG9uIHZhbCBieQp0cmFpbl9zb2xkZXIucHkpOyAtLWNvbmYgWCB1c2VzIG9uZSB0aHJlc2hvbGQgWCBmb3IgZXZlcnkgY2xhc3MgaW5zdGVhZC4KLS1sYWJlbHMgPGZvbGRlciBvZiBZT0xPIC50eHQ+OiBhbHNvIGRyYXdzIHRoZSB0cnVlIGJveGVzICh0aGluIGdyZWVuKSB0byBjb21wYXJlIHdpdGggdGhlIHByZWRpY3Rpb25zLgoKICBweXRob24gcHJlZGljdC5weSAtLXdlaWdodHMgYmVzdC5wdCAtLXNvdXJjZSBteV9ib2FyZC5qcGcgLS1vdXQgcmVzdWx0cwoiIiIKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgY3YyCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNodmlzaW9uCmZyb20gdWx0cmFseXRpY3MgaW1wb3J0IFlPTE8KCkNPTCA9IFsoMCwgMCwgMjU1KSwgKDI1NSwgMTI4LCAwKSwgKDI1NSwgMCwgMjU1KSwgKDAsIDIxNSwgMjU1KSwgKDI1NSwgMjU1LCAwKSwgKDEyOCwgMCwgMjU1KSwgKDAsIDEyOCwgMjU1KSwKICAgICAgICgxODAsIDEwNSwgMjU1KSwgKDQwLCAxODAsIDEyMCldCkdUID0gKDYwLCAyMjAsIDYwKQoKCmRlZiB0aWxlcyh3LCBoLCB0LCBvdmVybGFwKToKICAgIHN0ZXAgPSBpbnQodCAqICgxIC0gb3ZlcmxhcCkpCiAgICB4cyA9IGxpc3QocmFuZ2UoMCwgbWF4KDEsIHcgLSB0KSwgc3RlcCkpICsgW21heCgwLCB3IC0gdCldCiAgICB5cyA9IGxpc3QocmFuZ2UoMCwgbWF4KDEsIGggLSB0KSwgc3RlcCkpICsgW21heCgwLCBoIC0gdCldCiAgICByZXR1cm4gc29ydGVkKHsoeCwgeSkgZm9yIHggaW4geHMgZm9yIHkgaW4geXN9KQoKCmRlZiBwcmVkaWN0X2ltYWdlKG1vZGVsLCBpbWcsIGEsIGNvbmZfbWluKToKICAgIGgsIHcgPSBpbWcuc2hhcGVbOjJdCiAgICBqb2JzID0gWygwLCAwLCBpbWcpXSBpZiBtYXgoaCwgdykgPD0gYS50aWxlX2Fib3ZlIGVsc2UgXAogICAgICAgIFsoeCwgeSwgaW1nW3k6eSArIGEudGlsZSwgeDp4ICsgYS50aWxlXSkgZm9yIHgsIHkgaW4gdGlsZXModywgaCwgYS50aWxlLCBhLm92ZXJsYXApXQogICAgYm94ZXMsIHNjb3JlcywgY2xhc3NlcyA9IFtdLCBbXSwgW10KICAgIGZvciBrIGluIHJhbmdlKDAsIGxlbihqb2JzKSwgYS5iYXRjaCk6CiAgICAgICAgY2h1bmsgPSBqb2JzW2s6ayArIGEuYmF0Y2hdCiAgICAgICAgcmVzID0gbW9kZWwucHJlZGljdChbY1syXSBmb3IgYyBpbiBjaHVua10sIGltZ3N6PWEuaW1nc3osIGNvbmY9Y29uZl9taW4sIGlvdT1hLmlvdSwgdmVyYm9zZT1GYWxzZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRldmljZT1hLmRldmljZSkKICAgICAgICBmb3IgKHgwLCB5MCwgXyksIHIgaW4gemlwKGNodW5rLCByZXMpOgogICAgICAgICAgICBiID0gci5ib3hlcwogICAgICAgICAgICBpZiBsZW4oYik6CiAgICAgICAgICAgICAgICBib3hlcy5hcHBlbmQoYi54eXh5LmNwdSgpICsgdG9yY2gudGVuc29yKFt4MCwgeTAsIHgwLCB5MF0pKQogICAgICAgICAgICAgICAgc2NvcmVzLmFwcGVuZChiLmNvbmYuY3B1KCkpCiAgICAgICAgICAgICAgICBjbGFzc2VzLmFwcGVuZChiLmNscy5jcHUoKSkKICAgIGlmIG5vdCBib3hlczoKICAgICAgICByZXR1cm4gW10KICAgIEIsIFMsIEMgPSB0b3JjaC5jYXQoYm94ZXMpLCB0b3JjaC5jYXQoc2NvcmVzKSwgdG9yY2guY2F0KGNsYXNzZXMpCiAgICBrZWVwID0gdG9yY2h2aXNpb24ub3BzLmJhdGNoZWRfbm1zKEIsIFMsIEMsIGEuaW91KSBpZiBsZW4oam9icykgPiAxIGVsc2UgdG9yY2guYXJhbmdlKGxlbihTKSkKICAgIHJldHVybiBbKGludChDW2ldKSwgZmxvYXQoU1tpXSksICpbZmxvYXQodikgZm9yIHYgaW4gQltpXV0pIGZvciBpIGluIGtlZXBdCgoKZGVmIHJlYWRfZ3QobGJsLCB3LCBoKToKICAgIG91dCA9IFtdCiAgICBpZiBsYmwgYW5kIGxibC5leGlzdHMoKToKICAgICAgICBmb3IgbG4gaW4gbGJsLnJlYWRfdGV4dChlbmNvZGluZz0idXRmOCIpLnNwbGl0KCJcbiIpOgogICAgICAgICAgICBwID0gbG4uc3BsaXQoKQogICAgICAgICAgICBpZiBsZW4ocCkgPT0gNToKICAgICAgICAgICAgICAgIGMsIHgsIHksIGJ3LCBiaCA9IGludChwWzBdKSwgKm1hcChmbG9hdCwgcFsxOl0pCiAgICAgICAgICAgICAgICBvdXQuYXBwZW5kKChjLCAoeCAtIGJ3IC8gMikgKiB3LCAoeSAtIGJoIC8gMikgKiBoLCAoeCArIGJ3IC8gMikgKiB3LCAoeSArIGJoIC8gMikgKiBoKSkKICAgIHJldHVybiBvdXQKCgpkZWYgZHJhdyhpbWcsIGRldHMsIGd0LCBuYW1lcyk6CiAgICB2aXMgPSBpbWcuY29weSgpCiAgICB0aCA9IG1heCgyLCBtYXgoaW1nLnNoYXBlWzoyXSkgLy8gNTAwKQogICAgZm9yIGMsIHgwLCB5MCwgeDEsIHkxIGluIGd0OgogICAgICAgIGN2Mi5yZWN0YW5nbGUodmlzLCAoaW50KHgwKSwgaW50KHkwKSksIChpbnQoeDEpLCBpbnQoeTEpKSwgR1QsIG1heCgxLCB0aCAvLyAyKSwgY3YyLkxJTkVfQUEpCiAgICBmcyA9IDAuNDUgKiB0aAogICAgZm9yIGMsIHMsIHgwLCB5MCwgeDEsIHkxIGluIHNvcnRlZChkZXRzLCBrZXk9bGFtYmRhIGQ6IGRbMV0pOgogICAgICAgIGNvbCA9IENPTFtjICUgbGVuKENPTCldCiAgICAgICAgcDAsIHAxID0gKGludCh4MCkgLSB0aCwgaW50KHkwKSAtIHRoKSwgKGludCh4MSkgKyB0aCwgaW50KHkxKSArIHRoKQogICAgICAgIGN2Mi5yZWN0YW5nbGUodmlzLCBwMCwgcDEsIGNvbCwgdGgsIGN2Mi5MSU5FX0FBKQogICAgICAgIHR4dCA9IGYie25hbWVzW2NdfSB7czouMmZ9IgogICAgICAgICh0dywgdGh0KSwgXyA9IGN2Mi5nZXRUZXh0U2l6ZSh0eHQsIGN2Mi5GT05UX0hFUlNIRVlfU0lNUExFWCwgZnMsIG1heCgxLCB0aCAvLyAyKSkKICAgICAgICB0eSA9IHAwWzFdIC0gNCBpZiBwMFsxXSAtIHRodCAtIDggPiAwIGVsc2UgcDFbMV0gKyB0aHQgKyA2CiAgICAgICAgY3YyLnJlY3RhbmdsZSh2aXMsIChwMFswXSwgdHkgLSB0aHQgLSA0KSwgKHAwWzBdICsgdHcgKyA0LCB0eSArIDMpLCBjb2wsIC0xKQogICAgICAgIGN2Mi5wdXRUZXh0KHZpcywgdHh0LCAocDBbMF0gKyAyLCB0eSksIGN2Mi5GT05UX0hFUlNIRVlfU0lNUExFWCwgZnMsICgyNTUsIDI1NSwgMjU1KSwgbWF4KDEsIHRoIC8vIDIpLCBjdjIuTElORV9BQSkKICAgIGlmIGd0OgogICAgICAgIGN2Mi5wdXRUZXh0KHZpcywgImdyZWVuID0gdHJ1ZSBkZWZlY3QgKGxhYmVsKSIsICg4LCAxMiArIDE0ICogdGgpLCBjdjIuRk9OVF9IRVJTSEVZX1NJTVBMRVgsIGZzLCBHVCwKICAgICAgICAgICAgICAgICAgICBtYXgoMSwgdGggLy8gMiksIGN2Mi5MSU5FX0FBKQogICAgcmV0dXJuIHZpcwoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS13ZWlnaHRzIiwgcmVxdWlyZWQ9VHJ1ZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zb3VyY2UiLCByZXF1aXJlZD1UcnVlLCBoZWxwPSJpbWFnZSBmaWxlIG9yIGZvbGRlciIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgZGVmYXVsdD0icHJlZGljdGlvbnMiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWNvbmYiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PS0xLCBoZWxwPSJvbmUgdGhyZXNob2xkIGZvciBhbGwgY2xhc3NlcyAoZGVmYXVsdDogdGhyZXNob2xkcy5qc29uKSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdGhyZXNob2xkcyIsIGRlZmF1bHQ9Tm9uZSwgaGVscD0iZGVmYXVsdDogdGhyZXNob2xkcy5qc29uIG5leHQgdG8gdGhlIHdlaWdodHMiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWlvdSIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC41KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWltZ3N6IiwgdHlwZT1pbnQsIGRlZmF1bHQ9MCwgaGVscD0iMCA9IHRoZSB0cmFpbmluZyBpbWFnZSBzaXplIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS10aWxlIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTI4MCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1vdmVybGFwIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjI1KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRpbGUtYWJvdmUiLCB0eXBlPWludCwgZGVmYXVsdD0yMDAwLCBoZWxwPSJ0aWxlIHBob3RvcyB3aG9zZSBsb25nIHNpZGUgaXMgbGFyZ2VyIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1iYXRjaCIsIHR5cGU9aW50LCBkZWZhdWx0PTQpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbGFiZWxzIiwgZGVmYXVsdD1Ob25lLCBoZWxwPSJmb2xkZXIgd2l0aCBZT0xPIGxhYmVsIGZpbGVzOiBkcmF3IHRoZSB0cnVlIGJveGVzIHRvbyIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZGV2aWNlIiwgZGVmYXVsdD1Ob25lKQogICAgYSA9IGFwLnBhcnNlX2FyZ3MoKQogICAgbW9kZWwgPSBZT0xPKGEud2VpZ2h0cykKICAgIG5hbWVzID0gbW9kZWwubmFtZXMKICAgIGlmIG5vdCBhLmltZ3N6OgogICAgICAgIGEuaW1nc3ogPSBpbnQoKGdldGF0dHIobW9kZWwsICJja3B0IiwgTm9uZSkgb3Ige30pLmdldCgidHJhaW5fYXJncyIsIHt9KS5nZXQoImltZ3N6IiwgODAwKSkKICAgIHRocl9maWxlID0gUGF0aChhLnRocmVzaG9sZHMpIGlmIGEudGhyZXNob2xkcyBlbHNlIFBhdGgoYS53ZWlnaHRzKS53aXRoX25hbWUoInRocmVzaG9sZHMuanNvbiIpCiAgICBpZiBhLmNvbmYgPj0gMDoKICAgICAgICB0aHIgPSB7aTogYS5jb25mIGZvciBpIGluIG5hbWVzfQogICAgZWxpZiB0aHJfZmlsZS5leGlzdHMoKToKICAgICAgICBwZXIgPSBqc29uLmxvYWRzKHRocl9maWxlLnJlYWRfdGV4dChlbmNvZGluZz0idXRmOCIpKVsicGVyX2NsYXNzIl0KICAgICAgICB0aHIgPSB7aTogZmxvYXQocGVyLmdldChuLCAwLjI1KSkgZm9yIGksIG4gaW4gbmFtZXMuaXRlbXMoKX0KICAgICAgICBwcmludCgicGVyLWNsYXNzIHRocmVzaG9sZHM6Iiwge25hbWVzW2ldOiByb3VuZCh0LCAyKSBmb3IgaSwgdCBpbiB0aHIuaXRlbXMoKX0pCiAgICBlbHNlOgogICAgICAgIHRociA9IHtpOiAwLjI1IGZvciBpIGluIG5hbWVzfQogICAgc3JjID0gUGF0aChhLnNvdXJjZSkKICAgIGZpbGVzID0gc29ydGVkKHAgZm9yIHAgaW4gKHNyYy5pdGVyZGlyKCkgaWYgc3JjLmlzX2RpcigpIGVsc2UgW3NyY10pCiAgICAgICAgICAgICAgICAgICBpZiBwLnN1ZmZpeC5sb3dlcigpIGluIHsiLmpwZyIsICIuanBlZyIsICIucG5nIiwgIi5ibXAiLCAiLnRpZiIsICIudGlmZiIsICIud2VicCJ9KQogICAgb3V0ID0gUGF0aChhLm91dCkKICAgIG91dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBzdW1tYXJ5ID0gW10KICAgIGZvciBwIGluIGZpbGVzOgogICAgICAgIGltZyA9IGN2Mi5pbXJlYWQoc3RyKHApKQogICAgICAgIGlmIGltZyBpcyBOb25lOgogICAgICAgICAgICBwcmludCgiY2Fubm90IHJlYWQiLCBwKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGRldHMgPSBbZCBmb3IgZCBpbiBwcmVkaWN0X2ltYWdlKG1vZGVsLCBpbWcsIGEsIG1pbih0aHIudmFsdWVzKCkpKSBpZiBkWzFdID49IHRocltkWzBdXV0KICAgICAgICBndCA9IHJlYWRfZ3QoUGF0aChhLmxhYmVscykgLyAocC5zdGVtICsgIi50eHQiKSwgaW1nLnNoYXBlWzFdLCBpbWcuc2hhcGVbMF0pIGlmIGEubGFiZWxzIGVsc2UgW10KICAgICAgICBjdjIuaW13cml0ZShzdHIob3V0IC8gZiJ7cC5zdGVtfV9hbm5vdGF0ZWQuanBnIiksIGRyYXcoaW1nLCBkZXRzLCBndCwgbmFtZXMpKQogICAgICAgIChvdXQgLyBmIntwLnN0ZW19Lmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoCiAgICAgICAgICAgIFt7ImNsYXNzIjogbmFtZXNbY10sICJjb25mIjogcm91bmQocywgNCksICJib3giOiBbcm91bmQodiwgMSkgZm9yIHYgaW4gKHgwLCB5MCwgeDEsIHkxKV19CiAgICAgICAgICAgICBmb3IgYywgcywgeDAsIHkwLCB4MSwgeTEgaW4gZGV0c10sIGluZGVudD0xKSkKICAgICAgICBjbnQgPSB7bjogc3VtKDEgZm9yIGQgaW4gZGV0cyBpZiBkWzBdID09IGkpIGZvciBpLCBuIGluIG5hbWVzLml0ZW1zKCl9CiAgICAgICAgc3VtbWFyeS5hcHBlbmQoeyJpbWFnZSI6IHAubmFtZSwgImRlZmVjdHMiOiBsZW4oZGV0cyksICoqKHsidHJ1ZV9ib3hlcyI6IGxlbihndCl9IGlmIGEubGFiZWxzIGVsc2Uge30pLCAqKmNudH0pCiAgICAgICAgcHJpbnQoZiJ7cC5uYW1lfToge2xlbihkZXRzKX0gZGVmZWN0cyIgKyAoZiIgKHRydWU6IHtsZW4oZ3QpfSkiIGlmIGEubGFiZWxzIGVsc2UgIiIpICsKICAgICAgICAgICAgICBmIiB7KHtrOiB2IGZvciBrLCB2IGluIGNudC5pdGVtcygpIGlmIHZ9KX0iKQogICAgaWYgc3VtbWFyeToKICAgICAgICB3aXRoIG9wZW4ob3V0IC8gInN1bW1hcnkuY3N2IiwgInciLCBuZXdsaW5lPSIiLCBlbmNvZGluZz0idXRmOCIpIGFzIGY6CiAgICAgICAgICAgIHdyID0gY3N2LkRpY3RXcml0ZXIoZiwgZmllbGRuYW1lcz1saXN0KHN1bW1hcnlbMF0ua2V5cygpKSkKICAgICAgICAgICAgd3Iud3JpdGVoZWFkZXIoKQogICAgICAgICAgICB3ci53cml0ZXJvd3Moc3VtbWFyeSkKICAgIHByaW50KCJyZXN1bHRzIC0+Iiwgb3V0KQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK", "panel.py": "..."}
for name, blob in SCRIPTS.items():
    with open(f"{WORK}/{name}", "wb") as f:
        f.write(base64.b64decode(blob))
sys.path.insert(0, WORK)
import panel
importlib.reload(panel)
from IPython.display import display, HTML, Image
print("کدها آماده:", ", ".join(SCRIPTS))

ANSI = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]")


def sysinfo():
    out = []
    try:
        q = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu",
                            "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=5).stdout
        u, mu, mt, t = [x.strip() for x in q.strip().split(",")]
        out.append(f"GPU {u}% busy · {float(mu) / 1024:.1f}/{float(mt) / 1024:.0f} GB · {t}°C")
    except Exception:
        pass
    try:
        import psutil
        vm = psutil.virtual_memory()
        out.append(f"CPU {psutil.cpu_percent():.0f}% · RAM {vm.used / 1e9:.1f}/{vm.total / 1e9:.1f} GB")
    except Exception:
        pass
    return " | ".join(out)


def train_live(cmd, status_file, log_file):
    """Run train_solder.py and redraw the live panel every 2 s. Returns (exit code, last log lines, status)."""
    if os.path.exists(status_file):
        os.remove(status_file)
    tail = collections.deque(maxlen=14)
    with open(log_file, "a", encoding="utf-8") as lf:
        lf.write("\n$ " + " ".join(cmd) + "\n")
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})

    def reader():                              # reads the output as it comes: \r progress bars included
        buf, t_bar = "", 0.0
        with open(log_file, "a", encoding="utf-8") as lf:
            while True:
                chunk = p.stdout.read1(65536)
                if not chunk:
                    break
                buf += chunk.decode("utf-8", "replace")
                parts = re.split(r"(\r|\n)", buf)
                buf = parts.pop()
                for seg, sep in zip(parts[0::2], parts[1::2]):
                    s = ANSI.sub("", seg).strip()
                    if not s:
                        continue
                    is_bar = "━" in s or "─" in s
                    if is_bar and tail and ("━" in tail[-1] or "─" in tail[-1]):
                        tail[-1] = s           # the same progress bar, redrawn
                    else:
                        tail.append(s)
                    if not is_bar or sep == "\n" or time.time() - t_bar > 30:
                        lf.write(s + "\n")
                        lf.flush()
                        t_bar = time.time() if is_bar else t_bar

    th = threading.Thread(target=reader, daemon=True)
    th.start()

    def status():
        try:
            return json.load(open(status_file, encoding="utf-8"))
        except Exception:
            return {"phase": "setup", "run": RUN_NAME, "model": MODEL_SIZE, "imgsz": RUN_IMGSZ, "epochs": RUN_EPOCHS,
                    "session_start": t0}

    t0 = time.time()
    h = display(HTML(panel.live_html(status(), list(tail))), display_id=True) or PanelFile(f"{WORK}/live_panel.html")
    n_hist, chart_html, info, t_info = -1, "", "", 0.0
    try:
        while True:
            rc = p.poll()
            st = status()
            hist = st.get("history") or []
            if len(hist) != n_hist:
                n_hist, chart_html = len(hist), panel.chart(hist, st.get("epochs"))
            if time.time() - t_info > 5:
                info, t_info = sysinfo(), time.time()
            if rc is not None:
                th.join(10)
                if rc != 0 and st.get("phase") not in ("stopped", "done"):
                    st["phase"] = "error"
            h.update(HTML(panel.live_html(st, list(tail), info, chart_html)))
            if rc is not None:
                return rc, list(tail), st
            time.sleep(2)
    except KeyboardInterrupt:
        print("در حال توقف ... آخرین epoch کامل روی Drive ذخیره می‌شود (حداکثر یک دقیقه)", flush=True)
        p.send_signal(signal.SIGINT)
        for _ in range(3):
            try:
                p.wait(timeout=90)
                break
            except (subprocess.TimeoutExpired, KeyboardInterrupt):
                p.kill()
        th.join(10)
        st = status()
        st["phase"] = "stopped"
        h.update(HTML(panel.live_html(st, list(tail), info, chart_html)))
        print("متوقف شد. برای ادامه: Run all (یا فقط بلوک‌ها را از اول اجرا کن).")
        raise
    except BaseException:                      # a problem in this loop must never leave the training running alone
        p.kill()
        raise


class PanelFile:
    """Outside a notebook display() returns nothing: the panel goes to an HTML file instead."""
    def __init__(self, path):
        self.path = path

    def update(self, obj):
        with open(self.path, "w", encoding="utf-8") as f:
            f.write('<meta charset="utf-8">' + obj.data)

## 2) داده: پیدا کردن zip در Drive، باز کردن روی دیسک سریع، بررسی

In [ ]:
DATA = f"{WORK}/final_{MODEL}"
if os.path.exists(f"{DATA}/data.yaml"):
    print("دیتاست از قبل باز شده:", DATA)
else:
    name = f"{MODEL}_data.zip"
    src = f"{DRIVE_DIR}/{name}"
    if not os.path.exists(src):                  # search the rest of the Drive
        # a folder shared with you lives under "Shared with me": add it to My Drive as a shortcut, then
        # Colab sees it in MyDrive or under .shortcut-targets-by-id
        roots = [DRIVE_DIR, "/content/drive/MyDrive", "/content/drive/.shortcut-targets-by-id",
                 "/content/drive/Shareddrives"] if IN_COLAB else [DRIVE_DIR]
        hits = []
        for r in roots:                           # look up to 6 folders deep, nearest first
            for d in range(7):
                hits += glob.glob(r + "/*" * d + "/" + name)
            if hits:
                break
        if not hits:
            raise SystemExit(f"فایل {name} در Google Drive پیدا نشد. اگر پوشه برایت شیر شده: در Drive روی پوشه "
                             f"راست‌کلیک ▸ Organize ▸ Add shortcut ▸ My Drive، بعد دوباره Run all.")
        src = hits[0]
    size = os.path.getsize(src) / 1e9
    if shutil.disk_usage(WORK).free / 1e9 < size * 2.2 + 10:
        raise SystemExit(f"فضای دیسک Colab کم است: {size * 2.2 + 10:.0f} GB لازم است")
    print("داده پیدا شد:", src)
    open(f"{WORK}/.drive_folder", "w").write(os.path.dirname(src))   # results are saved next to the data
    print(f"کپی ({size:.1f} GB) روی دیسک سریع ...", flush=True)
    local = f"{WORK}/{name}"
    shutil.copyfile(src, local)
    print("باز کردن ...", flush=True)
    with zipfile.ZipFile(local) as z:
        z.extractall(WORK)
    os.remove(local)
    assert os.path.exists(f"{DATA}/data.yaml"), f"zip خراب است یا مال مدل دیگری است: {src}"
    print("باز شد:", DATA)

if not os.path.exists(f"{DATA}/.verified"):
    run_live([sys.executable, f"{WORK}/verify_dataset.py", "--root", DATA])
    open(f"{DATA}/.verified", "w").write("ok")
else:
    print("دیتاست قبلاً بررسی شده")

# results go next to the data; if that shared folder is read-only for you, into your own Drive
base = open(f"{WORK}/.drive_folder").read().strip() if os.path.exists(f"{WORK}/.drive_folder") else DRIVE_DIR
try:
    os.makedirs(f"{base}/runs", exist_ok=True)
    open(f"{base}/runs/.write_test", "w").write("ok")
except OSError:
    base = "/content/drive/MyDrive/PCB_Defect_Detection_results" if IN_COLAB else f"{WORK}/results"
    print("پوشه‌ی داده فقط-خواندنی است؛ نتیجه‌ها اینجا ذخیره می‌شوند:", base)
RUNS = f"{base}/runs_test" if QUICK_TEST else f"{base}/runs"
PROJECT = f"{RUNS}/{MODEL}"
os.makedirs(PROJECT, exist_ok=True)
print("نتیجه‌ها روی Drive:", PROJECT)

# keep the best weights of every OLDER run in saved_weights/ (never overwritten, never deleted)
SAVED = f"{base}/saved_weights"
for old in sorted(glob.glob(f"{PROJECT}/*/weights/best.pt")):
    run = os.path.basename(os.path.dirname(os.path.dirname(old)))
    if run == RUN_NAME:
        continue
    dst = f"{SAVED}/{MODEL}_{run}_best.pt"
    if not os.path.exists(dst):
        os.makedirs(SAVED, exist_ok=True)
        shutil.copyfile(old, dst)
        try:
            ep = torch.load(old, map_location="cpu", weights_only=False).get("epoch", -1) + 1
        except Exception:
            ep = "?"
        print(f"وزن بهترین epoch آموزش قبلی ({run}, epoch {ep}) ذخیره شد: {dst}")
    else:
        print("وزن آموزش قبلی از قبل ذخیره است:", dst)

## 3) وضعیت آموزش
قبل از شروع نشان می‌دهد چه چیزی روی Drive ذخیره شده: چند epoch انجام شده، بهترین دقت تا الان، زمان هر epoch،
زمان باقی‌مانده، و این‌که بلوک بعد آموزش را **از کدام epoch** ادامه می‌دهد. هر وقت خواستی فقط همین بلوک را اجرا کن
(بعد از بلوک‌های ۱ و ۲).

In [ ]:
display(HTML(panel.status_html(PROJECT, RUN_NAME, {"model": MODEL_SIZE, "imgsz": RUN_IMGSZ, "epochs": RUN_EPOCHS,
                                                   "patience": PATIENCE})))

## 4) آموزش — پنل زنده
پنل هر ۲ ثانیه به‌روز می‌شود: **نوار کل آموزش** (epoch چندم از چند، زمان باقی‌مانده)، **نوار epoch جاری**
(batch چندم، سرعت، loss)، مرحله‌ی ارزیابی روی val، جدول و نمودار دقت هر epoch، آخرین نسخه‌ی امن روی Drive، و وضعیت GPU.

اولین بار در هر جلسه چند دقیقه «آماده‌سازی» طول می‌کشد (کوچک کردن عکس‌های بزرگ و کش کردن تصاویر روی دیسک).
گزارش کامل: `/content/work/train_log.txt` (در پنل: «آخرین خط‌های گزارش»).

In [ ]:
status_file, log_file = f"{WORK}/live_status.json", f"{WORK}/train_log.txt"
try:                                           # a training left over from an earlier stopped cell: stop it cleanly
    if subprocess.run(["pkill", "-INT", "-f", "train_solder.py"], capture_output=True).returncode == 0:
        print("یک آموزش قبلی هنوز در حال اجرا بود؛ متوقفش می‌کنم ...", flush=True)
        for _ in range(60):
            time.sleep(2)
            if subprocess.run(["pgrep", "-f", "train_solder.py"], capture_output=True).returncode != 0:
                break
        subprocess.run(["pkill", "-KILL", "-f", "train_solder.py"], capture_output=True)
except FileNotFoundError:
    pass
cmd =[sys.executable, f"{WORK}/train_solder.py", "--data", DATA, "--work", f"{WORK}/runs", "--drive", PROJECT,
       "--model", MODEL_SIZE, "--imgsz", str(RUN_IMGSZ), "--epochs", str(RUN_EPOCHS), "--patience", str(PATIENCE),
       "--status", status_file, "--save-every", "1" if QUICK_TEST else str(SAVE_EVERY)]
if QUICK_TEST:
    cmd += ["--fraction", "0.03", "--cache", "False"]
cmd += ["--workers", str(max(2, min(16, os.cpu_count() or 2)))]           # every CPU core loads data
if not torch.cuda.is_available():                                           # no GPU: train on the CPU
    cmd += ["--device", "cpu"]
batch = int(BATCH)
for attempt in range(3):
    log_start = os.path.getsize(log_file) if os.path.exists(log_file) else 0
    rc, tail, st = train_live(cmd + (["--batch", str(batch)] if batch else []), status_file, log_file)
    if rc == 0:
        break
    with open(log_file, encoding="utf-8", errors="replace") as f:
        f.seek(log_start)
        log = f.read()
    if "out of memory" in log.lower() and attempt < 2:      # resume with a smaller batch
        batch = max(2, int(st.get("batch") or 8) // 2)
        print(f"حافظه‌ی GPU کم آمد؛ ادامه با batch {batch} ...", flush=True)
        continue
    print("\n".join(log.strip().splitlines()[-25:]))
    raise RuntimeError(f"آموزش با خطا تمام شد (کد {rc}) - خط‌های بالا علت را نوشته‌اند")
if QUICK_TEST:
    print("\nآزمایش سریع بی‌خطا تمام شد. در تنظیمات QUICK_TEST را خاموش کن و دوباره Run all بزن تا آموزش اصلی شروع شود.")

## 5) نتیجه
متریک‌ها روی **بردهایی که مدل هرگز ندیده** (test) حساب شده‌اند؛ برای هر کلاس جدا.
عکس‌های نمونه: کادر سبز نازک = عیب واقعی (برچسب)، کادر رنگی = پیش‌بینی مدل.

In [ ]:
FINAL = f"{PROJECT}/final"
if not os.path.exists(f"{FINAL}/summary.json"):
    print("هنوز آموزش تمام نشده")
else:
    s = json.load(open(f"{FINAL}/summary.json"))
    run = s["chosen"]
    ev = json.load(open(f"{PROJECT}/{run}/eval.json"))
    display(HTML(panel.eval_html(ev)))
    if len(s["runs"]) > 1:
        print("همه‌ی آموزش‌ها (mAP50-95):", json.dumps(s["runs"]), "| انتخاب‌شده با val:", run)
    for pat, w in [("results.png", 950), ("test_eval/confusion_matrix_normalized.png", 720), ("test_eval/*PR_curve.png", 720)]:
        for f in sorted(glob.glob(f"{PROJECT}/{run}/{pat}"))[:1]:
            display(Image(f, width=w))
    out = f"{WORK}/test_preview_{run}"
    if not glob.glob(out + "/*_annotated.jpg"):
        by_src = collections.defaultdict(list)       # test photos with defects, a few from every source
        for p in sorted(glob.glob(f"{DATA}/images/test/*.jpg")):
            lbl = p.replace("/images/", "/labels/")[:-4] + ".txt"
            if os.path.exists(lbl) and os.path.getsize(lbl) > 0:
                by_src[os.path.basename(p).split("__")[0]].append(p)
        picks = []
        while len(picks) < 8 and any(by_src.values()):
            for k in list(by_src):
                if by_src[k] and len(picks) < 8:
                    lst = by_src[k]
                    picks.append(lst.pop(len(lst) // 2))
        prev = f"{WORK}/preview_src"
        shutil.rmtree(prev, ignore_errors=True)
        os.makedirs(prev)
        for p in picks:
            shutil.copy2(p, prev)
        run_live([sys.executable, f"{WORK}/predict.py", "--weights", f"{FINAL}/best.pt", "--source", prev,
                  "--out", out, "--labels", f"{DATA}/labels/test"])
    for p in sorted(glob.glob(out + "/*_annotated.jpg")):
        display(Image(p, width=620))
    print("بهترین مدل:", f"{FINAL}/best.pt", "| آستانه‌ها:", f"{FINAL}/thresholds.json")

## 6) تست روی عکس خودت
`IMAGE` خالی بماند تا دکمه‌ی آپلود بیاید، یا مسیر یک عکس/پوشه در Drive را بنویس.
`CONF` = 0 یعنی آستانه‌ی تنظیم‌شده‌ی هر کلاس؛ عدد بین 0 و 1 یعنی یک آستانه برای همه (کمتر = عیب‌های بیشتر و هشدار اشتباه بیشتر).

In [ ]:
IMAGE = ""   #@param {type:"string"}
CONF  = 0   #@param {type:"number"}
weights = f"{PROJECT}/final/best.pt"
if not os.path.exists(weights):
    print("اول آموزش باید تمام شود:", weights)
else:
    if not IMAGE and IN_COLAB:
        from google.colab import files
        up = files.upload(); IMAGE = f"{WORK}/upload"; os.makedirs(IMAGE, exist_ok=True)
        for n, b in up.items():
            open(f"{IMAGE}/{n}", "wb").write(b)
    if IMAGE:
        out = f"{WORK}/my_test_{MODEL}"
        run_live([sys.executable, f"{WORK}/predict.py", "--weights", weights, "--source", IMAGE, "--out", out]
                 + (["--conf", str(CONF)] if CONF else []))
        for p in sorted(glob.glob(out + "/*_annotated.jpg")):
            display(Image(p, width=1100))